[Markdown Cell]
# Notebook 02 — Model B: legal encoder-decoder pretraining

Model A is on hold. This notebook independently trains a **randomly initialized,
BART-style bidirectional encoder and causal decoder** on the Notebook 00 artifacts.
No pretrained model or external corpus is used. The default architecture has
**62,490,624 unique trainable parameters**, including one shared token embedding.
The same weights serve the encoder input, decoder input, and vocabulary output.

Run on **Kaggle T4 x2**. Data and tokenizer are read from
`/kaggle/input/cs781-preprocessed-data/`; all code, indices, logs, checkpoints, and
predictions are written under `/kaggle/working/model_b_run/`.
Attach the verified preprocessing dataset before running; raw HF files and secrets
are unnecessary here. The notebook embeds every runtime module, so uploading this
single `.ipynb` is sufficient. Each module is assembled in small cells because
`torch.multiprocessing.spawn` needs importable worker definitions.

References: [BART](https://arxiv.org/abs/1910.13461),
[T5](https://arxiv.org/abs/1910.10683), and the supplied capstone document and Phase-1
evaluation script in `resources/`. This architecture uses learned positions,
pre-LayerNorm, GELU, and encoder cross-attention; its training objective is
conditional continuation rather than T5 span corruption.


[Markdown Cell]
## 1. Environment and imports
Use Kaggle's CUDA-enabled PyTorch installation (>=2.5). Only the small tokenizer and progress packages are installed. T4 training uses FP16 autocast and a gradient scaler.


In [1]:
# [Code Cell]
%pip install -q "tokenizers==0.22.1" "tqdm==4.67.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# [Code Cell]
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import json
import os
import signal
import sys
import subprocess
import torch
import numpy as np
from tokenizers import Tokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("Visible GPUs:", torch.cuda.device_count())
module_parts: dict[str, list[str]] = {}

def append_module(filename: str, source: str) -> None:
    """Retain one readable module section; import only after assembly completes."""
    assert filename.endswith(".py") and "/" not in filename and "\\" not in filename
    module_parts.setdefault(filename, []).append(source)


Python: 3.12.13 PyTorch: 2.10.0+cu128
Visible GPUs: 2


[Markdown Cell]
## 2. Configuration and parameter accounting
The defaults use 6 encoder layers, 6 decoder layers, width 512, 8 attention heads,
and feed-forward width 2048. With $D=512$, $F=2048$, and $V=32768$:
\[
P=VD+(S_{\max}+T_{\max})D
+N_e(4D^2+2DF+9D+F)+N_d(8D^2+2DF+15D+F)+4D.
\]
The executable assertion counts shared Parameters once and enforces
$50{,}000{,}000<P<300{,}000{,}000$. Sequence lengths below are **BPE tokens**.
The 200M-token input corpus is processed in document windows; about one fifth
of a full window supplies decoder labels. Logs report actual supervised tokens.


In [3]:
# [Code Cell]
append_module('config.py', r'''from dataclasses import asdict, dataclass, replace
from pathlib import Path
import json
import math

RUNTIME_VERSION = "cs781-model-b-v1"
SPECIAL_TOKENS = ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")
SOURCES = ("supreme_court", "high_courts", "acts", "tribunals")


@dataclass(frozen=True)
class TrainConfig:
    """Architecture, continuation objective, and immutable training choices."""

    data_dir: str = "/kaggle/input/cs781-preprocessed-data"
    run_dir: str = "/kaggle/working/model_b_run"
    resume_dir: str | None = None  # Optional attached checkpoint dataset.
    vocab_size: int = 32_768
    d_model: int = 512
    n_heads: int = 8
    encoder_layers: int = 6
    decoder_layers: int = 6
    d_ff: int = 2_048
    max_source_tokens: int = 2_048
    max_decoder_tokens: int = 1_024
    dropout: float = 0.1
    activation_checkpointing: bool = True
    train_prefix_tokens: int = 1_024
    train_target_tokens: int = 256
    epochs: int = 3
    micro_batch_size: int = 2  # Per GPU.
    accumulation_steps: int = 8
    world_size: int = 2
    num_workers: int = 0  # Predictable memory; increase only after profiling.
    learning_rate: float = 3e-4
    min_lr_ratio: float = 0.1
    warmup_updates: int = 500
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    max_run_seconds: int = 37_800  # 10.5 hours; save at the next update boundary.
    min_free_bytes: int = 1 << 30
    max_checkpoint_bytes: int = 4 << 30
    seed: int = 781
    deterministic: bool = False
    generation_examples_per_source: int = 2
    generation_token_limit: int = 1_024

    def validate(self) -> None:
        """Reject unsupported dimensions, precision assumptions, and budgets."""
        assert self.vocab_size == 32_768
        assert self.d_model > 0 and self.d_model % self.n_heads == 0
        assert self.encoder_layers > 0 and self.decoder_layers > 0 and self.d_ff > 0
        assert 0 <= self.dropout < 1
        assert self.train_prefix_tokens >= 1 and self.train_target_tokens >= 1
        assert self.train_prefix_tokens + 1 <= self.max_source_tokens
        assert self.train_target_tokens <= self.max_decoder_tokens
        assert self.epochs > 0 and self.micro_batch_size > 0 and self.accumulation_steps > 0
        assert self.world_size == 2, "Production training requires T4 x2."
        assert self.num_workers >= 0 and self.checkpoint_every_steps == 500
        assert self.learning_rate > 0 and 0 < self.min_lr_ratio <= 1
        assert self.warmup_updates >= 0 and self.max_grad_norm > 0
        assert self.max_consecutive_overflows > 0 and self.max_run_seconds > 0
        assert self.min_free_bytes >= 0 and self.max_checkpoint_bytes > 0
        assert 0 < self.generation_token_limit <= self.max_decoder_tokens
        assert self.generation_examples_per_source > 0

    @property
    def block_tokens(self) -> int:
        return self.train_prefix_tokens + self.train_target_tokens

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        return cls(**json.loads(Path(path).read_text(encoding="utf-8")))

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2) + "\n", encoding="utf-8")


def unique_trainable_parameters(model) -> int:
    """Count each shared Parameter object once, including the tied vocabulary."""
    return sum(p.numel() for p in {id(p): p for p in model.parameters() if p.requires_grad}.values())


def assert_parameter_boundary(model) -> int:
    count = unique_trainable_parameters(model)
    assert 50_000_000 < count < 300_000_000, f"Parameter boundary violated: {count:,}"
    return count


''')


[Markdown Cell]
The preferred path remains `/kaggle/input/cs781-preprocessed-data/`. If your attached dataset has another name or retains ZIP folders such as `kaggle/working/downloaded_output/`, locate the unique complete export below `/kaggle/input/`. Its folder names do not refer to the live working directory. Empty binaries and incomplete duplicate exports are rejected. Multiple complete exports require an explicit `cfg.data_dir`; full checksum verification still follows.


In [4]:
# [Code Cell]
append_module('config.py', r'''REQUIRED_DATA_FILES = (
    "metadata.json", "tokenizer.json", "train.bin", "val.bin",
    "train_documents.jsonl", "validation_documents.jsonl", "validation_pairs.jsonl",
)


def complete_preprocessing_directory(directory: Path) -> bool:
    """Check names and receipt sizes without loading binaries or copying input data.

    ZIP exports may contain a second, incomplete downloaded_output directory.
    Empty binaries and incomplete exports must never win automatic discovery.
    Full SHA-256 verification still runs in the existing data-contract check.
    """
    try:
        if not all((directory / name).is_file() for name in REQUIRED_DATA_FILES):
            return False
        metadata_path = directory / "metadata.json"
        if not 0 < metadata_path.stat().st_size <= (4 << 20):
            return False
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if not isinstance(metadata, dict) or metadata.get("status") != "verified":
            return False
        if metadata.get("schema_version") != 1 or metadata.get("dtype") != "<u2":
            return False
        if metadata.get("vocab_size") != 32_768:
            return False
        for name, key in (("train.bin", "train_tokens"), ("val.bin", "val_tokens")):
            count = metadata[key]
            if type(count) is not int or count <= 0 or (directory / name).stat().st_size != count * 2:
                return False
        for name in REQUIRED_DATA_FILES[1:]:
            size = (directory / name).stat().st_size
            if size <= 0 or size != metadata["artifacts"][name]["bytes"]:
                return False
        return True
    except (OSError, ValueError, KeyError, TypeError):
        return False


def resolve_preprocessed_data_dir(requested: str | Path,
                                  input_root: str | Path = "/kaggle/input") -> Path:
    """Prefer a complete explicit folder, otherwise locate one unique mounted export.

    Multiple complete exports require an explicit choice; files are never copied
    from read-only Kaggle input into scarce working storage.
    """
    requested_path = Path(requested).expanduser()
    if complete_preprocessing_directory(requested_path):
        return requested_path.resolve()
    roots = [requested_path] if requested_path.is_dir() else []
    mounted_root = Path(input_root)
    if mounted_root.is_dir() and mounted_root not in roots:
        roots.append(mounted_root)
    candidates: set[Path] = set()
    inspected: set[Path] = set()
    for root in roots:
        for metadata_path in root.rglob("metadata.json"):
            directory = metadata_path.parent.resolve()
            inspected.add(directory)
            if complete_preprocessing_directory(directory):
                candidates.add(directory)
    if len(candidates) == 1:
        return next(iter(candidates))
    if len(candidates) > 1:
        options = "\n".join(f"  {path}" for path in sorted(candidates))
        raise ValueError("Multiple complete preprocessing exports found. Set cfg.data_dir "
                         f"to one exact folder:\n{options}")
    found = "\n".join(f"  {path}" for path in sorted(inspected)[:8]) or "  none"
    raise FileNotFoundError(
        f"No complete preprocessing export at {requested_path} or below {mounted_root}. "
        "Attach the tokenized dataset and select the folder containing metadata.json, "
        "tokenizer.json, both nonempty .bin files, and the three JSONL manifests. "
        f"Inspected metadata folders (incomplete or invalid):\n{found}"
    )


def resolve_training_config(cfg: TrainConfig,
                            input_root: str | Path = "/kaggle/input") -> TrainConfig:
    """Canonicalize the input path before saving config or spawning GPU workers."""
    resolved = resolve_preprocessed_data_dir(cfg.data_dir, input_root)
    print(f"Preprocessed data: {resolved}", flush=True)
    return replace(cfg, data_dir=str(resolved))

''')


In [5]:
# [Code Cell]
# Execute only the import-free configuration module in the notebook.
exec(compile("".join(module_parts["config.py"]), "model_b/config.py", "exec"), globals())

cfg = resolve_training_config(TrainConfig())  # Edit explicit fields here before the first training run.
cfg.validate()
OFFICIAL_TEST_JSON: str | None = None  # Optional attached Phase-1 test file.
print(json.dumps(asdict(cfg), indent=2))


Preprocessed data: /kaggle/input/datasets/shubhamjoshi1806/dataset-tokenized/kaggle/working/downloaded_output
{
  "data_dir": "/kaggle/input/datasets/shubhamjoshi1806/dataset-tokenized/kaggle/working/downloaded_output",
  "run_dir": "/kaggle/working/model_b_run",
  "resume_dir": null,
  "vocab_size": 32768,
  "d_model": 512,
  "n_heads": 8,
  "encoder_layers": 6,
  "decoder_layers": 6,
  "d_ff": 2048,
  "max_source_tokens": 2048,
  "max_decoder_tokens": 1024,
  "dropout": 0.1,
  "activation_checkpointing": true,
  "train_prefix_tokens": 1024,
  "train_target_tokens": 256,
  "epochs": 3,
  "micro_batch_size": 2,
  "accumulation_steps": 8,
  "world_size": 2,
  "num_workers": 0,
  "learning_rate": 0.0003,
  "min_lr_ratio": 0.1,
  "warmup_updates": 500,
  "weight_decay": 0.1,
  "beta1": 0.9,
  "beta2": 0.95,
  "adam_epsilon": 1e-08,
  "max_grad_norm": 1.0,
  "checkpoint_every_steps": 500,
  "max_consecutive_overflows": 8,
  "max_run_seconds": 37800,
  "min_free_bytes": 1073741824,
  "max_c

[Markdown Cell]
## 3. Attention and information flow
\[
\operatorname{Attn}(Q,K,V)=\operatorname{softmax}(QK^\top/\sqrt{d_h}+M)V.
\]
Encoder attention is bidirectional within the observed prefix. Decoder attention
allows only its past; cross-attention sees only encoder memory. Padding keys are
excluded. PyTorch SDPA interprets boolean **True as allowed**.
[SDPA documentation](https://docs.pytorch.org/docs/2.7/generated/torch.nn.functional.scaled_dot_product_attention.html).


In [6]:
# [Code Cell]
append_module('model.py', r'''from dataclasses import dataclass
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from .config import TrainConfig, assert_parameter_boundary

KeyValue = tuple[Tensor, Tensor]


class MultiHeadAttention(nn.Module):
    """Q: [B,L,D], memory: [B,S,D], allowed: bool[B,1,L,S] -> [B,L,D].

    SDPA boolean True means an allowed key. Dropout is disabled in evaluation.
    All projections are learned from scratch; cached tensors have [B,H,S,D/H].
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, hidden: Tensor) -> Tensor:
        batch, length, _ = hidden.shape
        return hidden.view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)

    def project_kv(self, memory: Tensor) -> KeyValue:
        return self.split_heads(self.k_proj(memory)), self.split_heads(self.v_proj(memory))

    def attend(self, query: Tensor, kv: KeyValue, allowed: Tensor | None) -> Tensor:
        q = self.split_heads(self.q_proj(query))
        output = F.scaled_dot_product_attention(
            q, kv[0], kv[1], attn_mask=allowed,
            dropout_p=self.dropout if self.training else 0.0,
        )
        output = output.transpose(1, 2).contiguous().flatten(2)
        return self.out_proj(output)

    def forward(self, query: Tensor, memory: Tensor, allowed: Tensor) -> Tensor:
        return self.attend(query, self.project_kv(memory), allowed)


''')


[Markdown Cell]
Pre-normalization stabilizes residual updates:
$h'=h+\operatorname{Attn}(\operatorname{LN}(h))$ and
$h''=h'+W_2\operatorname{GELU}(W_1\operatorname{LN}(h'))$.
Dropout regularizes attention probabilities and residual/FFN branches.


In [7]:
# [Code Cell]
append_module('model.py', r'''class FeedForward(nn.Module):
    """Position-wise GELU feed-forward: [B,L,D] -> [B,L,D]."""

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.up = nn.Linear(cfg.d_model, cfg.d_ff)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.down(self.dropout(F.gelu(self.up(hidden))))


class EncoderLayer(nn.Module):
    """Pre-LayerNorm bidirectional self-attention and GELU residual block."""

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.self_attention = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.feed_forward = FeedForward(cfg)
        self.norm1, self.norm2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, hidden: Tensor, allowed: Tensor) -> Tensor:
        normalized = self.norm1(hidden)
        hidden = hidden + self.dropout(self.self_attention(normalized, normalized, allowed))
        return hidden + self.dropout(self.feed_forward(self.norm2(hidden)))


''')


[Markdown Cell]
A decoder block first attends to past target tokens, then reads the observed legal prompt, then applies its feed-forward network. The one-token API retains per-layer self K/V and reuses cross K/V during generation.


In [8]:
# [Code Cell]
append_module('model.py', r'''class DecoderLayer(nn.Module):
    """Causal self-attention, prompt cross-attention, then GELU feed-forward.

    Inputs: hidden[B,T,D], memory[B,S,D], self/cross allowed-key masks.
    Output: hidden[B,T,D]. The one-token step caches self and cross K/V.
    """

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.self_attention = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.cross_attention = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.feed_forward = FeedForward(cfg)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, hidden: Tensor, memory: Tensor, self_allowed: Tensor,
                cross_allowed: Tensor) -> Tensor:
        normalized = self.norm1(hidden)
        hidden = hidden + self.dropout(self.self_attention(normalized, normalized, self_allowed))
        hidden = hidden + self.dropout(self.cross_attention(self.norm2(hidden), memory, cross_allowed))
        return hidden + self.dropout(self.feed_forward(self.norm3(hidden)))

    def step(self, hidden: Tensor, previous: KeyValue | None, cross_kv: KeyValue,
             cross_allowed: Tensor) -> tuple[Tensor, KeyValue]:
        """Decode one position; every cached self key belongs to its past."""
        normalized = self.norm1(hidden)
        current = self.self_attention.project_kv(normalized)
        kv = current if previous is None else (
            torch.cat((previous[0], current[0]), dim=2),
            torch.cat((previous[1], current[1]), dim=2),
        )
        hidden = hidden + self.dropout(self.self_attention.attend(normalized, kv, None))
        hidden = hidden + self.dropout(self.cross_attention.attend(self.norm2(hidden), cross_kv, cross_allowed))
        return hidden + self.dropout(self.feed_forward(self.norm3(hidden))), kv


''')


[Markdown Cell]
Learned source and target positions use separate tables. The vocabulary embedding is shared; the output projection uses its weight directly. Non-reentrant activation checkpointing saves GPU memory by recomputing layers during backward. [PyTorch checkpointing](https://docs.pytorch.org/docs/2.7/checkpoint.html).


In [9]:
# [Code Cell]
append_module('model.py', r'''class LegalEncoderDecoder(nn.Module):
    """Randomly initialized BART-style architecture for conditional continuation.

    encode(source_ids[B,S], source_mask[B,S]) -> memory[B,S,D]
    forward(source_ids, source_mask, decoder_ids[B,T], decoder_mask[B,T])
        -> unnormalized vocabulary logits[B,T,V].
    A single embedding Parameter serves both token inputs and the output head.
    This is a continuation objective, not a reproduction of T5 denoising.
    """

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.shared_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.source_positions = nn.Embedding(cfg.max_source_tokens, cfg.d_model)
        self.target_positions = nn.Embedding(cfg.max_decoder_tokens, cfg.d_model)
        self.encoder = nn.ModuleList(EncoderLayer(cfg) for _ in range(cfg.encoder_layers))
        self.decoder = nn.ModuleList(DecoderLayer(cfg) for _ in range(cfg.decoder_layers))
        self.encoder_norm, self.decoder_norm = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.apply(self.initialize_weights)
        self.parameter_count = assert_parameter_boundary(self)

    @staticmethod
    def initialize_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def embed(self, ids: Tensor, positions: nn.Embedding, offset: int = 0) -> Tensor:
        assert offset + ids.shape[1] <= positions.num_embeddings
        position_ids = torch.arange(offset, offset + ids.shape[1], device=ids.device)
        return self.dropout(self.shared_embedding(ids) + positions(position_ids)[None])

    def encode(self, source_ids: Tensor, source_mask: Tensor) -> Tensor:
        hidden = self.embed(source_ids, self.source_positions)
        allowed = source_mask[:, None, None, :]
        for layer in self.encoder:
            if self.training and self.cfg.activation_checkpointing:
                hidden = checkpoint(layer, hidden, allowed, use_reentrant=False)
            else:
                hidden = layer(hidden, allowed)
        return self.encoder_norm(hidden)

    def decode(self, decoder_ids: Tensor, decoder_mask: Tensor, memory: Tensor,
               source_mask: Tensor) -> Tensor:
        hidden = self.embed(decoder_ids, self.target_positions)
        length = decoder_ids.shape[1]
        causal = torch.ones((length, length), device=hidden.device, dtype=torch.bool).tril()
        self_allowed = causal[None, None] & decoder_mask[:, None, None, :]
        cross_allowed = source_mask[:, None, None, :]
        for layer in self.decoder:
            if self.training and self.cfg.activation_checkpointing:
                hidden = checkpoint(layer, hidden, memory, self_allowed, cross_allowed, use_reentrant=False)
            else:
                hidden = layer(hidden, memory, self_allowed, cross_allowed)
        return self.decoder_norm(hidden)

    def forward(self, source_ids: Tensor, source_mask: Tensor, decoder_ids: Tensor,
                decoder_mask: Tensor) -> Tensor:
        memory = self.encode(source_ids, source_mask)
        hidden = self.decode(decoder_ids, decoder_mask, memory, source_mask)
        return F.linear(hidden, self.shared_embedding.weight)


''')


[Markdown Cell]
The cache grows only to the configured decoder capacity. Incremental logits are checked against full teacher-forced decoding before training.


In [10]:
# [Code Cell]
append_module('model.py', r'''@dataclass
class DecoderCache:
    """Inference-only cache; one self/cross K/V pair for each decoder layer."""
    self_kv: list[KeyValue | None]
    cross_kv: list[KeyValue]
    position: int = 0


def prepare_decoder_cache(model: LegalEncoderDecoder, memory: Tensor) -> DecoderCache:
    return DecoderCache([None] * len(model.decoder),
                        [layer.cross_attention.project_kv(memory) for layer in model.decoder])


def decode_one_token(model: LegalEncoderDecoder, token_ids: Tensor, source_mask: Tensor,
                     cache: DecoderCache) -> Tensor:
    """token_ids[B,1] -> logits[B,V], updating the bounded decoder cache."""
    assert token_ids.shape[1] == 1 and not model.training
    hidden = model.embed(token_ids, model.target_positions, cache.position)
    for index, layer in enumerate(model.decoder):
        hidden, cache.self_kv[index] = layer.step(
            hidden, cache.self_kv[index], cache.cross_kv[index], source_mask[:, None, None, :],
        )
    cache.position += 1
    return F.linear(model.decoder_norm(hidden)[:, 0], model.shared_embedding.weight)
''')


[Markdown Cell]
## 4. Input verification and memory limits
Check Notebook 00's metadata schema, six special IDs, file sizes, SHA-256 receipts, and token bounds. Both binary files are scanned in bounded memmap slices. Only offsets and compact indices are added to working storage.


In [11]:
# [Code Cell]
append_module('data.py', r'''from functools import partial
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SPECIAL_TOKENS, SOURCES


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


def strict_object(pairs: list[tuple]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=strict_object,
                      parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))


def verify_data_contract(cfg: TrainConfig) -> dict:
    """Verify bounded-memory binaries, tokenizer identity, and published receipts."""
    root = Path(cfg.data_dir)
    metadata = read_json(root / "metadata.json")
    assert metadata["status"] == "verified" and metadata["schema_version"] == 1
    assert metadata["dtype"] == "<u2" and metadata["vocab_size"] == cfg.vocab_size
    assert metadata["train_tokens"] % 100 == 0 and metadata["val_tokens"] % 4 == 0
    assert metadata["training"]["source_tokens"] == dict(zip(
        SOURCES, [metadata["train_tokens"] * percent // 100 for percent in (35, 35, 15, 15)]))
    assert all(metadata["validation"][source]["cached_tokens"] == metadata["val_tokens"] // 4
               for source in SOURCES)
    required = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",
                "validation_documents.jsonl", "validation_pairs.jsonl")
    for name in tqdm(required, desc="Verify preprocessing receipts"):
        path, receipt = root / name, metadata["artifacts"][name]
        assert path.stat().st_size == receipt["bytes"], f"Size mismatch: {name}"
        assert sha256_file(path) == receipt["sha256"], f"Checksum mismatch: {name}"
    tokenizer = Tokenizer.from_file(str(root / "tokenizer.json"))
    ids = {token: tokenizer.token_to_id(token) for token in SPECIAL_TOKENS}
    assert tokenizer.get_vocab_size() == 32_768
    assert ids == metadata["special_token_ids"] == dict(zip(SPECIAL_TOKENS, range(6)))
    for name, count in (("train.bin", metadata["train_tokens"]), ("val.bin", metadata["val_tokens"])):
        assert count > 0 and (root / name).stat().st_size == count * 2
        cache = np.memmap(root / name, dtype="<u2", mode="r", shape=(count,))
        for start in range(0, count, 1_000_000):
            assert int(cache[start:start + 1_000_000].max()) < cfg.vocab_size
        del cache
    identity = {name: metadata["artifacts"][name]["sha256"] for name in required}
    identity["metadata.json"] = sha256_file(root / "metadata.json")
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    return {"fingerprint": fingerprint, "metadata": metadata, "special_ids": ids}


''')


[Markdown Cell]
Training windows follow the document manifest, preserve the cached source mixture, and never cross a document boundary. A one-token tail cannot form a prefix/target pair and is reported as dropped. Whole-document training and validation hashes must be disjoint.


In [12]:
# [Code Cell]
append_module('data.py', r'''def prepare_indices(cfg: TrainConfig, contract: dict) -> dict:
    """Index offsets, never copy the raw/token corpus and never join documents."""
    root, run = Path(cfg.data_dir), Path(cfg.run_dir)
    index_dir = run / "indices"
    index_dir.mkdir(parents=True, exist_ok=True)
    offsets, lengths, counts, source_indices = [], [], [], []
    train_hashes, validation_hashes = set(), set()
    source_tokens, retained_tokens, dropped_tokens = Counter(), Counter(), Counter()
    cursor = 0
    with (root / "train_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line, object_pairs_hook=strict_object)
            offset, length, source = row["cache_offset"], row["cache_length"], row["source"]
            assert source in SOURCES and offset == cursor and length > 0
            assert row["sha256"] not in train_hashes
            train_hashes.add(row["sha256"])
            full, remainder = divmod(length, cfg.block_tokens)
            count = full + int(remainder >= 2)
            if count:
                offsets.append(offset); lengths.append(length); counts.append(count)
                source_indices.append(SOURCES.index(source))
            source_tokens[source] += length
            dropped = remainder if remainder < 2 else 0
            dropped_tokens[source] += dropped
            retained_tokens[source] += length - dropped
            cursor += length
    assert cursor == contract["metadata"]["train_tokens"]
    assert dict(source_tokens) == contract["metadata"]["training"]["source_tokens"]
    with (root / "validation_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            digest = json.loads(line)["sha256"]
            assert digest not in validation_hashes
            validation_hashes.add(digest)
    assert train_hashes.isdisjoint(validation_hashes)
    cumulative = np.cumsum(np.asarray(counts, dtype=np.int64))
    assert len(cumulative) and int(cumulative[-1]) >= cfg.world_size * cfg.micro_batch_size
    train_path = index_dir / "train_index.npz"
    temporary = train_path.with_suffix(".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, offsets=np.array(offsets, dtype=np.int64),
                            lengths=np.array(lengths, dtype=np.int64), cumulative=cumulative,
                            sources=np.array(source_indices, dtype=np.uint8))
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, train_path)
    validation = prepare_validation_index(cfg, validation_hashes, index_dir)
    report = {"train_examples": int(cumulative[-1]), "cached_source_tokens": dict(source_tokens),
              "eligible_source_tokens": dict(retained_tokens), "one_token_tails_dropped": dict(dropped_tokens),
              "validation": validation, "data_fingerprint": contract["fingerprint"]}
    (index_dir / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report


''')


[Markdown Cell]
Validation uses all full 512-word prompts and 128-word references in `validation_pairs.jsonl`, even when the corresponding final `val.bin` slice was cropped to meet its exact quota. BPE capacity truncations are counted explicitly. Long prompts retain their recent tail; cropped targets receive no invented EOS.


In [13]:
# [Code Cell]
append_module('data.py', r'''def tokenize_pair(tokenizer: Tokenizer, prompt: str, reference: str,
                  cfg: TrainConfig) -> tuple[list[int], list[int], bool, bool]:
    """Separate prompt encoding prevents any target token entering the encoder."""
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    target_ids = tokenizer.encode(" " + reference, add_special_tokens=False).ids
    source_cropped = len(prompt_ids) > cfg.max_source_tokens - 1
    target_cropped = len(target_ids) + 1 > cfg.max_decoder_tokens
    source = [1] + prompt_ids[-(cfg.max_source_tokens - 1):]
    target = target_ids[:cfg.max_decoder_tokens] if target_cropped else target_ids + [2]
    assert source and target
    return source, target, source_cropped, target_cropped


def prepare_validation_index(cfg: TrainConfig, allowed_hashes: set[str], index_dir: Path) -> dict:
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    offsets, identifiers, statistics = [], set(), Counter()
    with (Path(cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
        while True:
            offset = stream.tell()
            line = stream.readline()
            if not line:
                break
            row = json.loads(line, object_pairs_hook=strict_object)
            assert row["id"] not in identifiers and row["document_sha256"] in allowed_hashes
            assert len(row["prompt"].split()) == 512 and len(row["reference"].split()) == 128
            assert row["source"] in SOURCES
            identifiers.add(row["id"]); offsets.append(offset)
            _, _, source_crop, target_crop = tokenize_pair(tokenizer, row["prompt"], row["reference"], cfg)
            statistics[row["source"]] += 1
            statistics["source_truncations"] += int(source_crop)
            statistics["target_truncations"] += int(target_crop)
    assert offsets and all(statistics[source] for source in SOURCES)
    path = index_dir / "validation_offsets.npy"
    with path.with_suffix(".partial").open("wb") as stream:
        np.save(stream, np.asarray(offsets, dtype=np.int64), allow_pickle=False)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(path.with_suffix(".partial"), path)
    return dict(statistics, examples=len(offsets))


''')


[Markdown Cell]
Each process opens a read-only uint16 memmap lazily. A window uses up to 1024 observed prefix tokens and 256 unseen target tokens. The source receives BOS; an EOS label appears only if present at a true cached document end. Worker state carries neither an open memmap nor a tokenizer object.


In [14]:
# [Code Cell]
append_module('data.py', r'''class ContinuationDataset(Dataset):
    """One document window -> bidirectional prefix + unseen causal continuation."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg, self.cache = cfg, None
        with np.load(Path(cfg.run_dir) / "indices/train_index.npz", allow_pickle=False) as index:
            self.offsets, self.lengths = index["offsets"].copy(), index["lengths"].copy()
            self.cumulative = index["cumulative"].copy()

    def __len__(self) -> int:
        return int(self.cumulative[-1])

    def __getstate__(self) -> dict:
        return dict(self.__dict__, cache=None)

    def __getitem__(self, index: int) -> dict:
        assert 0 <= index < len(self)
        if self.cache is None:
            self.cache = np.memmap(Path(self.cfg.data_dir) / "train.bin", dtype="<u2", mode="r")
        document = int(np.searchsorted(self.cumulative, index, side="right"))
        previous_count = int(self.cumulative[document - 1]) if document else 0
        within = (index - previous_count) * self.cfg.block_tokens
        length = min(self.cfg.block_tokens, int(self.lengths[document]) - within)
        start = int(self.offsets[document]) + within
        tokens = np.asarray(self.cache[start:start + length], dtype=np.int64).tolist()
        prefix = min(self.cfg.train_prefix_tokens, length - 1,
                     max(1, (length * self.cfg.train_prefix_tokens + self.cfg.block_tokens - 1)
                         // self.cfg.block_tokens))
        source, target = [1] + tokens[:prefix], tokens[prefix:]
        assert 0 < len(target) <= self.cfg.train_target_tokens
        # Do not invent EOS at a cropped document or arbitrary block boundary.
        return {"source": source, "target": target}


class ValidationDataset(Dataset):
    """Read one complete held-out JSONL example; val.bin may have a cropped tail."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg = cfg
        self.offsets = np.load(Path(cfg.run_dir) / "indices/validation_offsets.npy", allow_pickle=False)
        self.tokenizer = None

    def __len__(self) -> int:
        return len(self.offsets)

    def __getstate__(self) -> dict:
        return dict(self.__dict__, tokenizer=None)

    def __getitem__(self, index: int) -> dict:
        if self.tokenizer is None:
            self.tokenizer = Tokenizer.from_file(str(Path(self.cfg.data_dir) / "tokenizer.json"))
        with (Path(self.cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
            stream.seek(int(self.offsets[index]))
            row = json.loads(stream.readline())
        source, target, _, _ = tokenize_pair(self.tokenizer, row["prompt"], row["reference"], self.cfg)
        return {"source": source, "target": target}


''')


[Markdown Cell]
Shift decoder inputs right: $[\mathrm{BOS},y_1,\ldots,y_{T-1}]$ predicts
$[y_1,\ldots,y_T]$. Padding labels are $-100$. A seeded global permutation is
partitioned into disjoint rank batches. Resume starts directly at `next_batch`,
without reading or training earlier batches. The last incomplete global batch
is dropped and its maximum size is documented.


In [15]:
# [Code Cell]
append_module('data.py', r'''def collate_continuations(rows: list[dict]) -> dict[str, torch.Tensor]:
    """Pad sources and targets; labels[t] is predicted from BOS,target[:t]."""
    source_length = max(len(row["source"]) for row in rows)
    target_length = max(len(row["target"]) for row in rows)
    source = torch.zeros((len(rows), source_length), dtype=torch.long)
    decoder = torch.zeros((len(rows), target_length), dtype=torch.long)
    labels = torch.full_like(decoder, -100)
    source_mask = torch.zeros_like(source, dtype=torch.bool)
    decoder_mask = torch.zeros_like(decoder, dtype=torch.bool)
    for index, row in enumerate(rows):
        s, t = len(row["source"]), len(row["target"])
        source[index, :s] = torch.tensor(row["source"])
        decoder[index, :t] = torch.tensor([1] + row["target"][:-1])
        labels[index, :t] = torch.tensor(row["target"])
        source_mask[index, :s] = True
        decoder_mask[index, :t] = True
    return {"source_ids": source, "source_mask": source_mask, "decoder_ids": decoder,
            "decoder_mask": decoder_mask, "labels": labels}


class EpochBatchSampler(Sampler):
    """Seeded global permutation with disjoint ranks and a direct batch cursor.

    No prefix batches are loaded on resume. At most world_size*batch_size-1
    examples are dropped per epoch; no duplicate examples pad the final batch.
    """

    def __init__(self, size: int, batch_size: int, rank: int, world_size: int,
                 seed: int, epoch: int, start_batch: int = 0) -> None:
        self.batch_size, self.rank, self.world_size = batch_size, rank, world_size
        self.num_batches = size // (world_size * batch_size)
        assert 0 <= start_batch <= self.num_batches and 0 <= rank < world_size
        self.start_batch = start_batch
        generator = torch.Generator().manual_seed(seed + epoch)
        count = self.num_batches * world_size * batch_size
        self.batches = torch.randperm(size, generator=generator)[:count].reshape(
            self.num_batches, world_size, batch_size,
        )[:, rank]

    def __iter__(self):
        for index in range(self.start_batch, self.num_batches):
            yield self.batches[index].tolist()

    def __len__(self) -> int:
        return self.num_batches - self.start_batch
''')


[Markdown Cell]
## 5. Checkpoint integrity and reproducibility
Data, learning settings, runtime source, and PyTorch version form a compatibility fingerprint. Changed training settings fail explicitly. Output paths and time/storage limits can change when moving a saved run. A Linux OS lock blocks concurrent trainers and releases automatically after process death.


In [16]:
# [Code Cell]
append_module('state.py', r'''from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import hashlib
import json
import math
import os
import random
import shutil
import numpy as np
import torch
import torch.distributed as dist
from .config import TrainConfig, RUNTIME_VERSION
from .data import sha256_file, read_json


def distributed() -> bool:
    return dist.is_available() and dist.is_initialized()


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n"); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)


def runtime_fingerprint(cfg: TrainConfig, contract: dict) -> str:
    """Paths and time/space limits may change; learning and data choices may not."""
    choices = asdict(cfg)
    for key in ("data_dir", "run_dir", "resume_dir", "max_run_seconds", "min_free_bytes",
                "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit"):
        choices.pop(key)
    sources = {path.name: sha256_file(path) for path in sorted(Path(__file__).parent.glob("*.py"))}
    payload = {"version": RUNTIME_VERSION, "config": choices,
               "data": contract["fingerprint"], "sources": sources,
               "torch": str(torch.__version__)}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


@contextmanager
def exclusive_run_lock(run_dir: Path):
    """Linux advisory lock is released by the OS even after a killed kernel."""
    import fcntl  # Kaggle/Linux only; production does not silently fall back.
    run_dir.mkdir(parents=True, exist_ok=True)
    with (run_dir / ".training.lock").open("a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Another Model B training process owns this run directory.") from exc
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


''')


[Markdown Cell]
Every checkpoint stores Python, NumPy, CPU Torch, and each rank's CUDA RNG state. All saved values are tensors or primitive containers so loading uses `weights_only=True`. DataLoader and sampler generators are separate from dropout's RNG. Exact saved-step replay is supported; CUDA bitwise identity also depends on kernels and the runtime.


In [17]:
# [Code Cell]
append_module('state.py', r'''def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {"python": random.getstate(), "torch_cpu": torch.get_rng_state(),
            "numpy": {"kind": numpy_state[0], "keys": torch.tensor(numpy_state[1].astype(np.int64)),
                      "position": numpy_state[2], "has_gauss": numpy_state[3],
                      "cached_gaussian": numpy_state[4]},
            "torch_cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state["python"]); torch.set_rng_state(state["torch_cpu"])
    ns = state["numpy"]
    np.random.set_state((ns["kind"], ns["keys"].numpy().astype(np.uint32), ns["position"],
                         ns["has_gauss"], ns["cached_gaussian"]))
    if device.type == "cuda":
        torch.cuda.set_rng_state(state["torch_cuda"], device)


def cpu_tree(value):
    """Own a CPU snapshot, retaining only tensors and weights-only-safe primitives."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value


''')


[Markdown Cell]
Two alternating full recovery slots and one full best checkpoint keep storage bounded. SHA-256 receipts commit after atomic weight-file publication; a corrupt/incomplete latest slot falls back to an intact earlier slot. If none is intact, training stops. Only checkpoints matching the current run are resumed.


In [18]:
# [Code Cell]
append_module('state.py', r'''class CheckpointManager:
    """Two full recovery slots plus one full best checkpoint, each with SHA-256.

    The inactive slot is replaced first; its receipt commits last. A crash
    between those writes leaves the previously committed slot recoverable.
    Every full checkpoint includes optimizer, scaler, scheduler, and cursor.
    """

    def __init__(self, cfg: TrainConfig, fingerprint: str) -> None:
        self.cfg, self.fingerprint, self.generation = cfg, fingerprint, 0
        self.directory = Path(cfg.run_dir) / "checkpoints"
        self.directory.mkdir(parents=True, exist_ok=True)

    def latest(self) -> tuple[Path, dict] | None:
        roots = [self.directory]
        if self.cfg.resume_dir:
            roots.append(Path(self.cfg.resume_dir))
        candidates, seen, problems = [], False, []
        for root in roots:
            for name in ("slot_0.pt", "slot_1.pt", "best.pt"):
                path, receipt_path = root / name, root / (name + ".json")
                if not path.exists() and not receipt_path.exists():
                    continue
                seen = True
                try:
                    receipt = read_json(receipt_path)
                    assert isinstance(receipt, dict)
                    assert all(type(receipt[key]) is int for key in ("bytes", "step", "generation", "epoch"))
                    assert receipt["bytes"] > 0 and receipt["step"] >= 0
                    assert receipt["generation"] > 0 and receipt["epoch"] >= 0
                    assert path.stat().st_size == receipt["bytes"]
                    assert sha256_file(path) == receipt["sha256"]
                    candidates.append((path, receipt))
                except (OSError, ValueError, TypeError, KeyError, AssertionError) as exc:
                    problems.append(f"{name}: {type(exc).__name__}")
        if seen and not candidates:
            raise RuntimeError(f"No intact checkpoint; preserve files and inspect receipts. {problems}")
        if not candidates:
            if self.cfg.resume_dir:
                raise FileNotFoundError("resume_dir contains no committed checkpoints.")
            return None
        chosen = max(candidates, key=lambda item: (item[1]["step"], item[1]["generation"]))
        if chosen[1]["fingerprint"] != self.fingerprint:
            raise RuntimeError("Checkpoint data/configuration/runtime differs. Use a fresh run directory.")
        self.generation = chosen[1]["generation"]
        if problems and (not distributed() or dist.get_rank() == 0):
            print(f"Ignoring incomplete checkpoints: {problems}; recovered step {chosen[1]['step']}", flush=True)
        return chosen

    def ensure_space(self, estimated_bytes: int) -> None:
        owned = sum(path.stat().st_size for path in self.directory.iterdir() if path.is_file())
        free = shutil.disk_usage(self.directory).free
        if free < estimated_bytes + self.cfg.min_free_bytes:
            raise OSError("Insufficient disk space for an atomic checkpoint and free-space reserve.")
        if owned + estimated_bytes > self.cfg.max_checkpoint_bytes:
            raise OSError("Checkpoint storage cap reached. Inspect this run's checkpoint files.")

    def restore(self, model, optimizer, scaler, scheduler, device: torch.device) -> dict | None:
        chosen = self.latest()
        if chosen is None:
            return None
        path, receipt = chosen
        payload = torch.load(path, map_location="cpu", weights_only=True)
        assert payload["fingerprint"] == self.fingerprint
        assert payload["step"] == receipt["step"] and payload["epoch"] == receipt["epoch"]
        assert payload["world_size"] == self.cfg.world_size
        model.load_state_dict(payload["model"], strict=True)
        optimizer.load_state_dict(payload["optimizer"])
        scaler.load_state_dict(payload["scaler"])
        scheduler.load_state_dict(payload["scheduler"])
        assert scheduler.steps == payload["cursor"]["optimizer_steps"]
        return payload


''')


[Markdown Cell]
Rank 0 saves the model, optimizer, FP16 scaler, scheduler, epoch, global step, next batch, epoch statistics, and all rank RNG states. Every rank receives save success/failure. Free-space and owned checkpoint limits are checked before serialization. Best checkpoints retain the complete state too.


In [19]:
# [Code Cell]
append_module('state.py', r'''    def save(self, model, optimizer, scaler, scheduler, cursor: dict,
             device: torch.device, rank: int, best: bool = False) -> None:
        local_rng = capture_rng(device)
        rng_states = [None] * dist.get_world_size() if distributed() else [local_rng]
        if distributed():
            dist.all_gather_object(rng_states, local_rng)
        outcome = [None]
        if rank == 0:
            try:
                self.generation += 1
                path = self.directory / f"slot_{self.generation % 2}.pt"
                temporary = path.with_name(path.name + ".partial")
                # FP32 weights + Adam's two FP32 moments + serialization headroom.
                estimated = sum(p.numel() for p in model.parameters()) * 12 + (32 << 20)
                self.ensure_space(estimated)
                payload = {"version": RUNTIME_VERSION, "fingerprint": self.fingerprint,
                           "world_size": self.cfg.world_size, "config": asdict(self.cfg),
                           "epoch": int(cursor["epoch"]), "step": int(cursor["global_step"]),
                           "model": cpu_tree(model.state_dict()),
                           "optimizer": cpu_tree(optimizer.state_dict()), "scaler": scaler.state_dict(),
                           "scheduler": scheduler.state_dict(), "cursor": dict(cursor),
                           "rng_by_rank": rng_states}
                with temporary.open("wb") as stream:
                    torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
                receipt = {"bytes": temporary.stat().st_size, "sha256": sha256_file(temporary),
                           "fingerprint": self.fingerprint, "generation": self.generation,
                           "epoch": payload["epoch"], "step": payload["step"],
                           "validation_nll": (cursor["best_validation_nll"]
                                              if math.isfinite(cursor["best_validation_nll"]) else None)}
                os.replace(temporary, path)
                atomic_json(path.with_name(path.name + ".json"), receipt)
                if best:
                    self.publish_best(path, receipt)
                outcome[0] = {"ok": True, "generation": self.generation}
            except Exception as exc:
                outcome[0] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        if distributed():
            dist.broadcast_object_list(outcome, src=0)
        if not outcome[0]["ok"]:
            raise RuntimeError(f"Checkpoint failed: {outcome[0]['error']}")
        self.generation = outcome[0]["generation"]

    def publish_best(self, source: Path, receipt: dict) -> None:
        """Hard-link the complete immutable snapshot; copy if links are unavailable."""
        path = self.directory / "best.pt"
        temporary = self.directory / "best.pt.partial"
        if temporary.exists():
            temporary.unlink()
        try:
            os.link(source, temporary)
        except OSError:
            self.ensure_space(source.stat().st_size)
            shutil.copyfile(source, temporary)
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
        os.replace(temporary, path)
        atomic_json(path.with_name(path.name + ".json"), receipt)


''')


[Markdown Cell]
Warm-up raises the LR; cosine decay approaches $0.1\eta_{\max}$.
The scheduler advances only after a successful optimizer update. An AMP overflow
consumes a completed batch group but does not advance optimizer or scheduler state.


In [20]:
# [Code Cell]
append_module('state.py', r'''class WarmupCosineScheduler:
    """LR advances only after a successful optimizer update, including resume."""

    def __init__(self, optimizer, total_steps: int, warmup_steps: int,
                 peak_lr: float, minimum_ratio: float) -> None:
        self.optimizer, self.total_steps = optimizer, total_steps
        self.warmup_steps = min(warmup_steps, max(0, total_steps - 1))
        self.peak_lr, self.minimum_ratio, self.steps = peak_lr, minimum_ratio, 0
        self.apply_lr()

    def apply_lr(self) -> None:
        if self.warmup_steps and self.steps < self.warmup_steps:
            ratio = (self.steps + 1) / self.warmup_steps
        else:
            denominator = max(1, self.total_steps - self.warmup_steps - 1)
            progress = min(1.0, max(0.0, (self.steps - self.warmup_steps) / denominator))
            ratio = self.minimum_ratio + (1 - self.minimum_ratio) * (1 + math.cos(math.pi * progress)) / 2
        for group in self.optimizer.param_groups:
            group["lr"] = self.peak_lr * ratio

    def step(self) -> None:
        self.steps += 1
        self.apply_lr()

    def state_dict(self) -> dict:
        return {key: getattr(self, key) for key in (
            "total_steps", "warmup_steps", "peak_lr", "minimum_ratio", "steps")}

    def load_state_dict(self, state: dict) -> None:
        for key in ("total_steps", "warmup_steps", "peak_lr", "minimum_ratio"):
            assert state[key] == getattr(self, key), f"Scheduler mismatch: {key}"
        self.steps = state["steps"]
        self.apply_lr()
''')


[Markdown Cell]
## 6. Distributed optimization and validation
The objective matches Phase 1 continuation:
\[
\mathcal L=-\frac{1}{N_y}\sum_{(x,y)}\sum_t\log p_\theta(y_t\mid y_{<t},\operatorname{Encoder}(x)).
\]
AdamW excludes bias/norm vectors from decay. Matrix parameters, including the
shared vocabulary, decay once. FP32 loss computation protects the vocabulary
softmax while FP16 autocast accelerates attention and projections on T4.


In [21]:
# [Code Cell]
append_module('train.py', r'''import argparse
from contextlib import nullcontext
from datetime import timedelta
from functools import partial
from pathlib import Path
import json
import math
import os
import signal
import socket
import time
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from .config import TrainConfig, assert_parameter_boundary, resolve_training_config
from .data import (verify_data_contract, prepare_indices, ContinuationDataset,
                   ValidationDataset, EpochBatchSampler, collate_continuations)
from .model import LegalEncoderDecoder
from .state import (CheckpointManager, WarmupCosineScheduler, atomic_json, distributed,
                    exclusive_run_lock, runtime_fingerprint, seed_everything, restore_rng)


def build_optimizer(model: nn.Module, cfg: TrainConfig):
    """Decay matrix weights once; exclude bias and normalization vectors."""
    decay, no_decay = [], []
    for parameter in model.parameters():
        if parameter.requires_grad:
            (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), eps=cfg.adam_epsilon,
        foreach=False,  # Avoid large optimizer temporary tensor lists on T4.
    )


def reduce_sum(tensor: torch.Tensor) -> torch.Tensor:
    if distributed():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor


def any_rank(flag: bool, device: torch.device) -> bool:
    value = torch.tensor(int(flag), device=device, dtype=torch.int32)
    if distributed():
        dist.all_reduce(value, op=dist.ReduceOp.MAX)
    return bool(value.item())


def move_batch(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device, non_blocking=device.type == "cuda") for key, value in batch.items()}


def autocast_context(device: torch.device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()


def summed_loss(model, batch: dict) -> torch.Tensor:
    logits = model(batch["source_ids"], batch["source_mask"], batch["decoder_ids"], batch["decoder_mask"])
    # Loss and vocabulary log-softmax stay in FP32 even when projections use FP16.
    return F.cross_entropy(logits.flatten(0, 1).float(), batch["labels"].flatten(),
                           ignore_index=-100, reduction="sum")


''')


[Markdown Cell]
With $R$ ranks, scale each microbatch's summed loss by $R/N_y$.
DDP's rank averaging then gives the exact token-weighted gradient across ranks,
variable target lengths, and the short final accumulation group. `no_sync()` wraps
both forward and backward for nonfinal microbatches. Unscale before clipping;
every rank skips and reduces its scale together on overflow.
[DDP](https://docs.pytorch.org/docs/2.7/notes/ddp.html),
[AMP accumulation](https://docs.pytorch.org/docs/2.7/notes/amp_examples.html).


In [22]:
# [Code Cell]
append_module('train.py', r'''def train_one_update(model, optimizer, scaler, scheduler, batches: list[dict],
                     cfg: TrainConfig, device: torch.device) -> dict:
    """One complete accumulation group; no checkpoint can contain partial gradients.

    DDP averages rank gradients. Scaling each summed loss by R / global_tokens
    yields the exact token-mean gradient across variable lengths and microbatches.
    """
    world_size = dist.get_world_size() if distributed() else 1
    count = sum(int((batch["labels"] != -100).sum()) for batch in batches)
    global_tokens = int(reduce_sum(torch.tensor(count, dtype=torch.int64, device=device)).item())
    assert global_tokens > 0
    optimizer.zero_grad(set_to_none=True)
    local_loss = torch.zeros((), dtype=torch.float64, device=device)
    used_lr = optimizer.param_groups[0]["lr"]
    for index, cpu_batch in enumerate(batches):
        batch = move_batch(cpu_batch, device)
        synchronize = index == len(batches) - 1
        context = model.no_sync() if isinstance(model, DDP) and not synchronize else nullcontext()
        with context:
            with autocast_context(device):
                loss_sum = summed_loss(model, batch)
            if any_rank(not bool(torch.isfinite(loss_sum).item()), device):
                raise FloatingPointError("Non-finite loss; recover from the last committed checkpoint.")
            local_loss += loss_sum.detach().double()
            scaler.scale(loss_sum * (world_size / global_tokens)).backward()
    scaler.unscale_(optimizer)
    gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm,
                                            error_if_nonfinite=False, foreach=False)
    overflow = any_rank(not bool(torch.isfinite(gradient_norm).item()), device)
    if overflow:
        if not scaler.is_enabled():
            raise FloatingPointError("Non-finite FP32 gradients.")
        # Every rank skips together; explicit scale change also resets scaler state.
        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())
    else:
        scaler.step(optimizer); scaler.update(); scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    loss_value = float(reduce_sum(local_loss).item())
    return {"loss_sum": loss_value, "tokens": global_tokens, "loss": loss_value / global_tokens,
            "learning_rate": used_lr, "overflow": overflow,
            "gradient_norm": float(gradient_norm.item())}


''')


[Markdown Cell]
Validation ranks visit disjoint examples without duplication. Use the raw module for unequal rank batch counts, then reduce total loss and target tokens once. Conditional perplexity is `exp(target NLL)`; it is specific to this continuation objective. Best selection uses held-out NLL. Official BERTScore/ROUGE/BLEU require the supplied evaluator and are not fabricated here.


In [23]:
# [Code Cell]
append_module('train.py', r'''@torch.inference_mode()
def evaluate(model: LegalEncoderDecoder, cfg: TrainConfig, device: torch.device,
             rank: int, world_size: int) -> dict:
    """Visit every held-out pair exactly once globally, without sampler padding."""
    model.eval()
    dataset = ValidationDataset(cfg)
    subset = Subset(dataset, range(rank, len(dataset), world_size))
    loader = DataLoader(subset, batch_size=cfg.micro_batch_size, shuffle=False,
                        collate_fn=collate_continuations, num_workers=0,
                        generator=torch.Generator().manual_seed(cfg.seed + 10_000),
                        pin_memory=device.type == "cuda")
    totals = torch.zeros(2, dtype=torch.float64, device=device)
    for cpu_batch in tqdm(loader, desc="Held-out continuation NLL", disable=rank != 0, mininterval=1):
        batch = move_batch(cpu_batch, device)
        with autocast_context(device):
            loss = summed_loss(model, batch)
        totals[0] += loss.double()
        totals[1] += (batch["labels"] != -100).sum()
    reduce_sum(totals)
    assert totals[1] > 0 and torch.isfinite(totals).all(), "Invalid validation loss"
    nll = float((totals[0] / totals[1]).item())
    model.train()
    return {"conditional_nll": nll, "conditional_perplexity": math.exp(min(nll, 80)),
            "perplexity_capped": nll > 80, "target_tokens": int(totals[1].item()),
            "examples": len(dataset)}


''')


[Markdown Cell]
The cursor records the next **rank-local microbatch**, completed update groups, and successful optimizer updates separately. Logs newer than a recovered checkpoint are removed before replay. Prefetch cannot change sample content because each dataset item is a deterministic function of its index.


In [24]:
# [Code Cell]
append_module('train.py', r'''def new_cursor() -> dict:
    return {"epoch": 0, "next_batch": 0, "global_step": 0, "optimizer_steps": 0,
            "epoch_loss_sum": 0.0, "epoch_tokens": 0, "consecutive_overflows": 0,
            "epoch_overflows": 0, "best_validation_nll": float("inf"), "history": []}


def training_loader(dataset, cfg: TrainConfig, epoch: int, start_batch: int, rank: int):
    sampler = EpochBatchSampler(len(dataset), cfg.micro_batch_size, rank, cfg.world_size,
                                cfg.seed, epoch, start_batch)
    loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_continuations,
                        num_workers=cfg.num_workers, pin_memory=True, persistent_workers=False,
                        generator=torch.Generator().manual_seed(cfg.seed + epoch * 31 + rank),
                        **({"multiprocessing_context": "spawn", "prefetch_factor": 1}
                           if cfg.num_workers else {}))
    return loader, sampler


def repair_step_log(path: Path, saved_step: int) -> None:
    """Remove interrupted/replayed log tails newer than the committed cursor."""
    if not path.exists():
        return
    temporary = path.with_name(path.name + ".partial")
    with path.open(encoding="utf-8") as source, temporary.open("w", encoding="utf-8") as target:
        for line in source:
            try:
                row = json.loads(line)
            except ValueError:
                break
            if row["step"] <= saved_step:
                target.write(line)
        target.flush(); os.fsync(target.fileno())
    os.replace(temporary, path)


''')


[Markdown Cell]
One spawned process owns one T4. DDP synchronizes parameters and gradients, while model/optimizer/scaler/scheduler/RNG state is restored independently on each rank. Only rank 0 prints progress and metrics. Incompatible or incomplete checkpoint state is rejected.


In [25]:
# [Code Cell]
append_module('train.py', r'''def worker(rank: int, cfg: TrainConfig, contract: dict, fingerprint: str,
           port: int, stop_event) -> None:
    """One spawned process per T4; all training collectives follow the same order."""
    os.environ["MASTER_ADDR"], os.environ["MASTER_PORT"] = "127.0.0.1", str(port)
    torch.cuda.set_device(rank)
    device = torch.device("cuda", rank)
    signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
    signal.signal(signal.SIGINT, lambda *_: stop_event.set())
    dist.init_process_group(backend="nccl", rank=rank, world_size=cfg.world_size,
                            timeout=timedelta(minutes=15))
    started = time.monotonic()
    try:
        seed_everything(cfg.seed + rank, cfg.deterministic)
        model = LegalEncoderDecoder(cfg).to(device)
        assert_parameter_boundary(model)
        ddp = DDP(model, device_ids=[rank], output_device=rank,
                  broadcast_buffers=False, find_unused_parameters=False,
                  gradient_as_bucket_view=True)
        optimizer = build_optimizer(model, cfg)
        dataset = ContinuationDataset(cfg)
        batches_per_epoch = len(dataset) // (cfg.world_size * cfg.micro_batch_size)
        updates_per_epoch = math.ceil(batches_per_epoch / cfg.accumulation_steps)
        scheduler = WarmupCosineScheduler(optimizer, updates_per_epoch * cfg.epochs,
                                         cfg.warmup_updates, cfg.learning_rate, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        manager = CheckpointManager(cfg, fingerprint)
        payload = manager.restore(model, optimizer, scaler, scheduler, device)
        cursor = new_cursor() if payload is None else payload["cursor"]
        assert 0 <= cursor["epoch"] <= cfg.epochs and 0 <= cursor["next_batch"] <= batches_per_epoch
        assert cursor["next_batch"] % cfg.accumulation_steps == 0 or cursor["next_batch"] == batches_per_epoch
        if payload is not None:
            assert len(payload["rng_by_rank"]) == cfg.world_size
            restore_rng(payload["rng_by_rank"][rank], device)
        log_path = Path(cfg.run_dir) / "training_steps.jsonl"
        if rank == 0:
            repair_step_log(log_path, cursor["global_step"])
            print(f"Model B: {model.parameter_count:,} unique trainable parameters", flush=True)
            print(f"Resume: epoch={cursor['epoch']} next_batch={cursor['next_batch']} "
                  f"step={cursor['global_step']} optimizer_step={cursor['optimizer_steps']}", flush=True)
        run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                            cursor, cfg, device, rank, stop_event, started, log_path)
    finally:
        dist.destroy_process_group()


''')


[Markdown Cell]
Checkpoints are automatic **every 500 completed accumulation groups** and **at every epoch end**. Epoch/step tqdm bars show loss and LR on rank 0. SIGINT/SIGTERM and the 10.5-hour run limit request a checkpoint at the next completed update. An abrupt kill can lose updates since the last save (at most 499 groups); recovery resumes exactly from the saved cursor. Preserve working files across Kaggle sessions.


In [26]:
# [Code Cell]
append_module('train.py', r'''def run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                        cursor: dict, cfg: TrainConfig, device: torch.device, rank: int,
                        stop_event, started: float, log_path: Path) -> None:
    epoch_bar = tqdm(total=cfg.epochs, initial=cursor["epoch"], desc="Model B epochs", disable=rank != 0)
    stopped = False
    try:
        while cursor["epoch"] < cfg.epochs:
            loader, sampler = training_loader(dataset, cfg, cursor["epoch"], cursor["next_batch"], rank)
            iterator = iter(loader)
            total_updates = math.ceil(sampler.num_batches / cfg.accumulation_steps)
            step_bar = tqdm(total=total_updates,
                            initial=math.ceil(cursor["next_batch"] / cfg.accumulation_steps),
                            desc=f"Epoch {cursor['epoch'] + 1}/{cfg.epochs}", disable=rank != 0,
                            mininterval=1, leave=False)
            try:
                while cursor["next_batch"] < sampler.num_batches:
                    count = min(cfg.accumulation_steps, sampler.num_batches - cursor["next_batch"])
                    batches = [next(iterator) for _ in range(count)]
                    metrics = train_one_update(ddp, optimizer, scaler, scheduler, batches, cfg, device)
                    cursor["next_batch"] += count
                    cursor["global_step"] += 1
                    cursor["optimizer_steps"] += int(not metrics["overflow"])
                    cursor["epoch_loss_sum"] += metrics["loss_sum"]
                    cursor["epoch_tokens"] += metrics["tokens"]
                    cursor["epoch_overflows"] += int(metrics["overflow"])
                    cursor["consecutive_overflows"] = cursor["consecutive_overflows"] + 1 if metrics["overflow"] else 0
                    assert cursor["optimizer_steps"] == scheduler.steps
                    if rank == 0:
                        with log_path.open("a", encoding="utf-8") as stream:
                            row = {"epoch": cursor["epoch"], "step": cursor["global_step"],
                                   "optimizer_step": cursor["optimizer_steps"],
                                   "loss": metrics["loss"], "learning_rate": metrics["learning_rate"],
                                   "tokens": metrics["tokens"], "overflow": metrics["overflow"]}
                            stream.write(json.dumps(row, allow_nan=False) + "\n")
                        step_bar.update(1)
                        step_bar.set_postfix(step=cursor["global_step"], loss=f"{metrics['loss']:.4f}",
                                             lr=f"{metrics['learning_rate']:.2e}")
                    stopping = any_rank(stop_event.is_set() or time.monotonic() - started >= cfg.max_run_seconds, device)
                    repeated_overflow = cursor["consecutive_overflows"] >= cfg.max_consecutive_overflows
                    if cursor["global_step"] % cfg.checkpoint_every_steps == 0 or stopping or repeated_overflow:
                        manager.save(model, optimizer, scaler, scheduler, cursor, device, rank)
                    if repeated_overflow:
                        raise FloatingPointError("Repeated AMP overflows; checkpoint saved. Inspect the scale/LR before resuming.")
                    if stopping:
                        stopped = True
                        break
            finally:
                step_bar.close()
                del iterator, loader
            if stopped:
                break
            validation = evaluate(model, cfg, device, rank, cfg.world_size)
            improved = validation["conditional_nll"] < cursor["best_validation_nll"]
            if improved:
                cursor["best_validation_nll"] = validation["conditional_nll"]
            record = {"epoch": cursor["epoch"] + 1,
                      "train_nll": cursor["epoch_loss_sum"] / cursor["epoch_tokens"],
                      "training_target_tokens": cursor["epoch_tokens"],
                      "overflows": cursor["epoch_overflows"], **validation}
            cursor["history"].append(record)
            cursor.update(epoch=cursor["epoch"] + 1, next_batch=0, epoch_loss_sum=0.0,
                          epoch_tokens=0, epoch_overflows=0)
            manager.save(model, optimizer, scaler, scheduler, cursor, device, rank, best=improved)
            if rank == 0:
                epoch_bar.update(1); epoch_bar.set_postfix(val_nll=f"{validation['conditional_nll']:.4f}")
                print(json.dumps(record), flush=True)
        if rank == 0:
            atomic_json(Path(cfg.run_dir) / "training_summary.json", {
                "status": "paused_at_saved_update" if stopped else "complete",
                "epoch": cursor["epoch"], "next_batch": cursor["next_batch"],
                "step": cursor["global_step"], "optimizer_steps": cursor["optimizer_steps"],
                "history": cursor["history"], "fingerprint": manager.fingerprint,
            })
    finally:
        epoch_bar.close()


''')


[Markdown Cell]
The production launcher requires exactly two T4 GPUs and NCCL, then uses `torch.multiprocessing.spawn`. It has no alternate production backend. If restoring from an attached checkpoint dataset, set `resume_dir` to the directory containing the `.pt` files and their `.json` receipts. Kaggle-discarded working files cannot be recovered.


In [27]:
# [Code Cell]
append_module('train.py', r'''def require_dual_t4() -> None:
    version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
    assert version >= (2, 5), "Use Kaggle's current PyTorch CUDA runtime (>=2.5)."
    assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select Kaggle accelerator T4 x2."
    assert dist.is_nccl_available(), "The PyTorch build needs NCCL support."
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        assert "T4" in name, f"Expected T4 on device {rank}: {name}"
        print(f"GPU {rank}: {name}, {torch.cuda.get_device_properties(rank).total_memory / 2**30:.1f} GiB")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config)); cfg.validate()
    require_dual_t4()
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    with exclusive_run_lock(Path(cfg.run_dir)):
        contract = verify_data_contract(cfg)
        report = prepare_indices(cfg, contract)
        fingerprint = runtime_fingerprint(cfg, contract)
        atomic_json(Path(cfg.run_dir) / "data_report.json", report)
        with torch.device("meta"):
            count = assert_parameter_boundary(LegalEncoderDecoder(cfg))
        print(f"Verified Model B parameter count: {count:,}", flush=True)
        # Uncommitted pipeline-owned temporaries cannot be restored.
        checkpoint_dir = Path(cfg.run_dir) / "checkpoints"
        if checkpoint_dir.exists():
            for name in ("slot_0.pt.partial", "slot_1.pt.partial", "best.pt.partial",
                         "slot_0.pt.json.partial", "slot_1.pt.json.partial", "best.pt.json.partial"):
                (checkpoint_dir / name).unlink(missing_ok=True)
        context = mp.get_context("spawn")
        stop_event = context.Event()
        signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
        signal.signal(signal.SIGINT, lambda *_: stop_event.set())
        with socket.socket() as sock:
            sock.bind(("127.0.0.1", 0)); port = sock.getsockname()[1]
        processes = mp.spawn(worker, args=(cfg, contract, fingerprint, port, stop_event),
                             nprocs=cfg.world_size, join=False)
        while not processes.join(timeout=1):
            pass


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
## 7. Generation and evaluator-compatible outputs
Greedy decoding uses cached self/cross K/V and generates an extra word so the 128th word is complete. It exports only the continuation. A token-cap failure is explicit; no repeated or invented filler completes an undersized output.


In [28]:
# [Code Cell]
append_module('infer.py', r'''import argparse
from pathlib import Path
import hashlib
import json
import os
import shutil
import sqlite3
import zipfile
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SOURCES, SPECIAL_TOKENS, resolve_training_config
from .data import read_json, sha256_file, verify_data_contract
from .model import LegalEncoderDecoder, prepare_decoder_cache, decode_one_token
from .state import CheckpointManager, atomic_json, runtime_fingerprint, seed_everything, exclusive_run_lock
from .train import autocast_context


@torch.inference_mode()
def generate_continuation(model: LegalEncoderDecoder, tokenizer: Tokenizer, prompt: str,
                          device: torch.device, token_limit: int, target_words: int = 128) -> str:
    """Generate until an extra word closes the requested final word; return 128.

    Token count is not word count. EOS/control tokens are suppressed; exhausting
    the decoder/token bound fails explicitly instead of inventing padding text.
    """
    model.eval()
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    ids = [1] + prompt_ids[-(model.cfg.max_source_tokens - 1):]
    source = torch.tensor([ids], device=device)
    source_mask = torch.ones_like(source, dtype=torch.bool)
    with autocast_context(device):
        memory = model.encode(source, source_mask)
        cache = prepare_decoder_cache(model, memory)
    token = torch.tensor([[1]], device=device)
    generated = []
    limit = min(token_limit, model.cfg.max_decoder_tokens)
    forbidden = [tokenizer.token_to_id(special) for special in SPECIAL_TOKENS]
    for step in range(limit):
        with autocast_context(device):
            logits = decode_one_token(model, token, source_mask, cache).float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite generation logits")
        logits[:, forbidden] = float("-inf")
        chosen = int(logits.argmax(dim=-1).item())
        generated.append(chosen)
        token = torch.tensor([[chosen]], device=device)
        if (step + 1) % 8 == 0 or step + 1 == limit:
            text = tokenizer.decode(generated, skip_special_tokens=True)
            words = text.split()
            if len(words) >= target_words + 1:
                continuation = " ".join(words[:target_words])
                assert len(continuation.split()) == target_words
                return continuation
    raise RuntimeError(f"Could not close {target_words} words within {limit} generated tokens. "
                       "Inspect training/generation quality or use a larger decoder capacity in a new experiment.")


''')


[Markdown Cell]
Prefer the verified best held-out checkpoint; if no epoch has completed, use the latest committed training checkpoint. The tokenizer and model configuration must match its fingerprint.


In [29]:
# [Code Cell]
append_module('infer.py', r'''def load_inference_model(cfg: TrainConfig, fingerprint: str, device: torch.device):
    manager = CheckpointManager(cfg, fingerprint)
    latest = manager.latest()
    if latest is None:
        raise FileNotFoundError("No trained checkpoint exists. Run distributed training first.")
    chosen = latest
    roots = [manager.directory] + ([Path(cfg.resume_dir)] if cfg.resume_dir else [])
    best_candidates = []
    for root in roots:
        path, receipt_path = root / "best.pt", root / "best.pt.json"
        if path.exists() and receipt_path.exists():
            try:
                receipt = read_json(receipt_path)
                assert receipt["fingerprint"] == fingerprint and path.stat().st_size == receipt["bytes"]
                assert sha256_file(path) == receipt["sha256"]
                score = receipt.get("validation_nll")
                assert score is None or isinstance(score, (int, float))
                best_candidates.append((path, receipt))
            except (OSError, ValueError, TypeError, KeyError, AssertionError):
                continue
    if best_candidates:
        chosen = min(best_candidates, key=lambda item: (
            item[1].get("validation_nll") if item[1].get("validation_nll") is not None else float("inf"),
            -item[1]["step"], -item[1]["generation"]))
    payload = torch.load(chosen[0], map_location="cpu", weights_only=True)
    assert payload["fingerprint"] == fingerprint
    model = LegalEncoderDecoder(cfg)
    model.load_state_dict(payload["model"], strict=True)
    model.to(device).eval()
    receipt = chosen[1]
    del payload
    return model, receipt


''')


[Markdown Cell]
The default development sample uses two examples per legal source. Optional official inference reads a user-attached test JSON with the exact string IDs and 512-word prompts. Private references are never accessed.


In [30]:
# [Code Cell]
append_module('infer.py', r'''def generation_records(cfg: TrainConfig, kind: str, input_json: str | None):
    """Use exact official string IDs; development records come only from held-out data."""
    if kind == "test":
        if input_json is None:
            raise ValueError("Provide --input-json pointing to the official Phase-1 test prompts.")
        payload = read_json(Path(input_json))
        assert isinstance(payload, dict) and payload
        for identifier, row in payload.items():
            assert isinstance(identifier, str) and isinstance(row, dict)
            assert isinstance(row.get("text"), str) and len(row["text"].split()) == 512
            yield {"id": identifier, "prompt": row["text"]}
    else:
        counts = dict.fromkeys(SOURCES, 0)
        with (Path(cfg.data_dir) / "validation_pairs.jsonl").open(encoding="utf-8") as stream:
            for line in stream:
                row = json.loads(line)
                source = row["source"]
                if counts[source] < cfg.generation_examples_per_source:
                    counts[source] += 1
                    yield row
        assert all(count == cfg.generation_examples_per_source for count in counts.values()), counts


def export_predictions(path: Path, predictions: dict) -> None:
    assert predictions and all(len(row["text"].split()) == 128 for row in predictions.values())
    atomic_json(path, predictions)


''')


[Markdown Cell]
Every completed prediction is committed to SQLite. Restart skips matching cached examples; a checkpoint or prompt change invalidates reuse. Official inference produces `prediction.json` and a ZIP containing that file at its root. These are local artifacts; nothing is submitted automatically.


In [31]:
# [Code Cell]
append_module('infer.py', r'''def run_generation(cfg: TrainConfig, kind: str, input_json: str | None = None,
                   device: torch.device | None = None) -> None:
    cfg.validate()
    if device is None:
        assert torch.cuda.is_available(), "Generation requires the Kaggle CUDA runtime."
        device = torch.device("cuda", 0)
    seed_everything(cfg.seed, cfg.deterministic)
    contract = verify_data_contract(cfg)
    fingerprint = runtime_fingerprint(cfg, contract)
    model, receipt = load_inference_model(cfg, fingerprint, device)
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    directory = Path(cfg.run_dir) / "generation" / kind
    directory.mkdir(parents=True, exist_ok=True)
    records = list(generation_records(cfg, kind, input_json))
    assert len({row["id"] for row in records}) == len(records)
    prompt_truncations = sum(len(tokenizer.encode(row["prompt"], add_special_tokens=False).ids)
                             > cfg.max_source_tokens - 1 for row in records)
    namespace = hashlib.sha256(json.dumps({"checkpoint": receipt["sha256"], "kind": kind,
                "tokens": cfg.generation_token_limit, "words": 128}, sort_keys=True).encode()).hexdigest()
    connection = sqlite3.connect(directory / "generation_cache.sqlite")
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=FULL")
    connection.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT, id TEXT, "
                       "prompt_sha TEXT, text TEXT, PRIMARY KEY(namespace,id))")
    predictions, truth = {}, {}
    try:
        for row in tqdm(records, desc=f"Model B {kind}: 128-word continuations"):
            prompt_sha = hashlib.sha256(row["prompt"].encode()).hexdigest()
            cached = connection.execute("SELECT prompt_sha,text FROM predictions WHERE namespace=? AND id=?",
                                        (namespace, row["id"])).fetchone()
            if cached is not None and cached[0] == prompt_sha and len(cached[1].split()) == 128:
                text = cached[1]
            else:
                if shutil.disk_usage(directory).free < cfg.min_free_bytes + (1 << 20):
                    raise OSError("Insufficient disk space for durable generation outputs.")
                text = generate_continuation(model, tokenizer, row["prompt"], device, cfg.generation_token_limit)
                connection.execute("INSERT OR REPLACE INTO predictions VALUES (?,?,?,?)",
                                   (namespace, row["id"], prompt_sha, text))
                connection.commit()  # Each completed example survives a later failure.
            predictions[row["id"]] = {"text": text}
            if "reference" in row:
                truth[row["id"]] = {"text": row["reference"]}
        prediction_path = directory / "prediction.json"
        export_predictions(prediction_path, predictions)
        if truth:
            assert truth.keys() == predictions.keys()
            atomic_json(directory / "truth.json", truth)
        if kind == "test":
            temporary = directory / "submission_model_b.zip.partial"
            with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
                archive.write(prediction_path, arcname="prediction.json")
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
            os.replace(temporary, directory / "submission_model_b.zip")
        atomic_json(directory / "generation_report.json", {
            "checkpoint_sha256": receipt["sha256"], "checkpoint_epoch": receipt["epoch"],
            "checkpoint_step": receipt["step"], "examples": len(predictions),
            "words_per_continuation": 128, "decoding": "greedy with cached self/cross K/V",
            "prompt_tail_limit": cfg.max_source_tokens - 1,
            "prompt_truncations": prompt_truncations,
            "token_limit": cfg.generation_token_limit, "data_fingerprint": contract["fingerprint"],
        })
        print(f"Saved {len(predictions)} continuations to {prediction_path}", flush=True)
    finally:
        connection.close()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--kind", choices=("validation", "test"), default="validation")
    parser.add_argument("--input-json")
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config))
    with exclusive_run_lock(Path(cfg.run_dir)):
        run_generation(cfg, args.kind, args.input_json)


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
## 8. Architecture sanity checks
The actual 6+6-layer architecture is exercised on short CPU sequences. Verify logit shape, causal isolation, bidirectional prompt flow, padding invariance, full versus cached decoding, teacher-forcing shifts, and gradients through encoder/cross-attention/shared vocabulary. The strict parameter assertion also runs before DDP initialization.


In [32]:
# [Code Cell]
append_module('sanity.py', r'''from dataclasses import replace
import gc
import torch
from torch.nn import functional as F
from .config import TrainConfig, assert_parameter_boundary
from .model import LegalEncoderDecoder, prepare_decoder_cache, decode_one_token
from .data import collate_continuations


def architecture_sanity(cfg: TrainConfig) -> dict:
    """Exercise the real architecture on short CPU sequences before starting DDP."""
    threads = torch.get_num_threads()
    torch.set_num_threads(min(4, threads))
    try:
        with torch.device("meta"):
            meta_model = LegalEncoderDecoder(cfg)
            count = assert_parameter_boundary(meta_model)
        del meta_model
        model = LegalEncoderDecoder(cfg).cpu().eval()
        source = torch.randint(6, cfg.vocab_size, (2, 12))
        decoder = torch.randint(6, cfg.vocab_size, (2, 6))
        source_mask, decoder_mask = torch.ones_like(source, dtype=torch.bool), torch.ones_like(decoder, dtype=torch.bool)
        source_mask[1, -3:] = False
        with torch.inference_mode():
            logits = model(source, source_mask, decoder, decoder_mask)
            assert logits.shape == (2, 6, cfg.vocab_size) and torch.isfinite(logits).all()
            changed = decoder.clone(); changed[:, 3:] = (changed[:, 3:] + 7) % cfg.vocab_size
            altered = model(source, source_mask, changed, decoder_mask)
            torch.testing.assert_close(logits[:, :3], altered[:, :3], rtol=1e-5, atol=1e-6)
            padded = source.clone(); padded[1, -3:] = 8
            torch.testing.assert_close(logits, model(padded, source_mask, decoder, decoder_mask), rtol=1e-5, atol=1e-6)
            memory = model.encode(source, source_mask)
            changed_source = source.clone(); changed_source[0, -1] = 9
            assert not torch.allclose(memory[0, 0], model.encode(changed_source, source_mask)[0, 0])
            cache = prepare_decoder_cache(model, memory)
            cached = torch.stack([decode_one_token(model, decoder[:, step:step + 1], source_mask, cache)
                                  for step in range(decoder.shape[1])], dim=1)
            torch.testing.assert_close(logits, cached, rtol=1e-4, atol=2e-6)
            cache_error = float((logits - cached).abs().max())
        batch = collate_continuations([{"source": [1, 6, 7], "target": [8, 9, 2]},
                                       {"source": [1, 6], "target": [7]}])
        assert batch["decoder_ids"][0].tolist() == [1, 8, 9]
        assert batch["labels"][1].tolist() == [7, -100, -100]
        model.train()
        output = model(batch["source_ids"], batch["source_mask"], batch["decoder_ids"], batch["decoder_mask"])
        loss = F.cross_entropy(output.flatten(0, 1), batch["labels"].flatten(), ignore_index=-100)
        loss.backward()
        for parameter in (model.shared_embedding.weight, model.encoder[0].self_attention.q_proj.weight,
                          model.decoder[0].cross_attention.q_proj.weight):
            assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
            assert parameter.grad.abs().sum() > 0
        report = {"parameters": count, "logit_shape": list(logits.shape),
                  "causal_mask": "passed", "source_padding": "passed", "bidirectional_encoder": "passed",
                  "cached_decoding_max_abs_error": cache_error, "gradient_flow": "passed"}
        del model, logits, output, loss
        gc.collect()
        return report
    finally:
        torch.set_num_threads(threads)
''')


[Markdown Cell]
## 9. Materialize the importable runtime
The following cell writes the embedded sections to an isolated package in working storage. It also saves the explicit configuration. The notebook does not depend on companion local files.


In [33]:
# [Code Cell]
code_dir = Path(cfg.run_dir) / "code"
package_dir = code_dir / "model_b"
package_dir.mkdir(parents=True, exist_ok=True)
module_parts["__init__.py"] = ['"""From-scratch CS-781 encoder-decoder and reproducible DDP training runtime."""\n']
for filename, sections in module_parts.items():
    source = "".join(sections)
    compile(source, filename, "exec")
    path = package_dir / filename
    temporary = path.with_suffix(".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(source); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
config_path = Path(cfg.run_dir) / "train_config.json"
cfg.save(config_path)
sys.path.insert(0, str(code_dir))
for module_name in list(sys.modules):
    if module_name == "model_b" or module_name.startswith("model_b."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("Importable package:", package_dir)
print("Training configuration:", config_path)


Importable package: /kaggle/working/model_b_run/code/model_b
Training configuration: /kaggle/working/model_b_run/train_config.json


In [34]:
# [Code Cell]
from model_b.sanity import architecture_sanity
sanity_report = architecture_sanity(cfg)
print(json.dumps(sanity_report, indent=2))
assert 50_000_000 < sanity_report["parameters"] < 300_000_000


{
  "parameters": 62490624,
  "logit_shape": [
    2,
    6,
    32768
  ],
  "causal_mask": "passed",
  "source_padding": "passed",
  "bidirectional_encoder": "passed",
  "cached_decoding_max_abs_error": 8.344650268554688e-07,
  "gradient_flow": "passed"
}


[Markdown Cell]
## 10. Launch and resume distributed training
The trainer verifies input checksums, builds compact indices, discovers compatible checkpoints, and spawns two NCCL workers. Output is forwarded live. Interrupting this cell asks the trainer to finish its current update and save; let the saving message finish before restarting. Re-running the cell resumes the latest intact local checkpoint automatically.


In [35]:
# [Code Cell]
def run_streamed(command: list[str], working_directory: Path) -> None:
    """Forward subprocess output, including tqdm carriage returns, in Jupyter."""
    import select
    import codecs
    process = subprocess.Popen(command, cwd=str(working_directory), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=0, env=os.environ.copy())
    interrupted = False
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    assert process.stdout is not None
    try:
        while True:
            try:
                ready, _, _ = select.select([process.stdout], [], [], 1.0)
                if ready:
                    chunk = os.read(process.stdout.fileno(), 8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                interrupted = True
                print("\nRequesting a checkpoint at the next update boundary; keep this cell running.")
                process.send_signal(signal.SIGINT)
        return_code = process.wait()
        sys.stdout.write(decoder.decode(b"", final=True)); sys.stdout.flush()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
        if interrupted:
            print("Trainer exited after the checkpoint request.")
    finally:
        process.stdout.close()

run_streamed([sys.executable, "-u", "-m", "model_b.train", "--config", str(config_path)], code_dir)


Preprocessed data: /kaggle/input/datasets/shubhamjoshi1806/dataset-tokenized/kaggle/working/downloaded_output
GPU 0: Tesla T4, 14.6 GiB
GPU 1: Tesla T4, 14.6 GiB
Verify preprocessing receipts: 100%|██████████| 6/6 [00:08<00:00,  1.37s/it]
Verified Model B parameter count: 62,490,624
Model B: 62,490,624 unique trainable parameters
Resume: epoch=0 next_batch=0 step=0 optimizer_step=0
Epoch 1/3: 100%|██████████| 8089/8089 [2:39:11<00:00,  1.11s/it, loss=2.8297, lr=2.38e-04, step=8089]
                                                                                                     
Model B epochs:  33%|███▎      | 1/3 [2:40:14<5:20:28, 9614.42s/it, val_nll=3.6522]{"epoch": 1, "train_nll": 3.5877464954897254, "training_target_tokens": 39935991, "overflows": 2, "conditional_nll": 3.652189409886128, "conditional_perplexity": 38.55899514503753, "perplexity_capped": false, "target_tokens": 991327, "examples": 6279}

Epoch 2/3: 100%|██████████| 8089/8089 [2:38:45<00:00,  1.16s/it, loss=1.906

In [36]:
# [Code Cell]
summary_path = Path(cfg.run_dir) / "training_summary.json"
if summary_path.exists():
    training_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(training_summary, indent=2))
data_report_path = Path(cfg.run_dir) / "data_report.json"
if data_report_path.exists():
    print(data_report_path.read_text(encoding="utf-8"))


{
  "status": "complete",
  "epoch": 3,
  "next_batch": 0,
  "step": 24267,
  "optimizer_steps": 24256,
  "history": [
    {
      "epoch": 1,
      "train_nll": 3.5877464954897254,
      "training_target_tokens": 39935991,
      "overflows": 2,
      "conditional_nll": 3.652189409886128,
      "conditional_perplexity": 38.55899514503753,
      "perplexity_capped": false,
      "target_tokens": 991327,
      "examples": 6279
    },
    {
      "epoch": 2,
      "train_nll": 2.821047039096085,
      "training_target_tokens": 39936150,
      "overflows": 5,
      "conditional_nll": 3.336377697175039,
      "conditional_perplexity": 28.117093392370275,
      "perplexity_capped": false,
      "target_tokens": 991327,
      "examples": 6279
    },
    {
      "epoch": 3,
      "train_nll": 2.626181752478497,
      "training_target_tokens": 39936117,
      "overflows": 4,
      "conditional_nll": 3.2213218031198574,
      "conditional_perplexity": 25.061224302423074,
      "perplexity_capped

[Markdown Cell]
## 11. Generate balanced development continuations
The GPU workers have exited before this separate single-GPU inference process begins. Compare these outputs with their held-out references using the supplied Phase-1 evaluator. If training quality cannot produce complete 128-word continuations within the token limit, generation stops with a clear error and keeps completed examples cached.


In [37]:
# [Code Cell]
run_streamed([sys.executable, "-u", "-m", "model_b.infer", "--config", str(config_path),
              "--kind", "validation"], code_dir)
validation_dir = Path(cfg.run_dir) / "generation/validation"
print("Development predictions:", validation_dir / "prediction.json")
print("Matching references:", validation_dir / "truth.json")


Preprocessed data: /kaggle/input/datasets/shubhamjoshi1806/dataset-tokenized/kaggle/working/downloaded_output
Model B validation: 128-word continuations: 100%|██████████| 8/8 [00:09<00:00,  1.24s/it]
Saved 8 continuations to /kaggle/working/model_b_run/generation/validation/prediction.json
Development predictions: /kaggle/working/model_b_run/generation/validation/prediction.json
Matching references: /kaggle/working/model_b_run/generation/validation/truth.json


[Markdown Cell]
## 12. Optional official inference
Set `OFFICIAL_TEST_JSON` in the configuration cell to the attached official `Phase1- Test.json`. IDs are copied exactly; the ZIP contains only `prediction.json`. Keep all local artifacts until you verify the file and decide to submit it yourself.


In [38]:
# [Code Cell]
if OFFICIAL_TEST_JSON is not None:
    test_path = Path(OFFICIAL_TEST_JSON)
    assert test_path.is_file(), test_path
    run_streamed([sys.executable, "-u", "-m", "model_b.infer", "--config", str(config_path),
                  "--kind", "test", "--input-json", str(test_path)], code_dir)
    print("Submission archive:", Path(cfg.run_dir) / "generation/test/submission_model_b.zip")
else:
    print("Official inference skipped; set OFFICIAL_TEST_JSON to enable it.")


Official inference skipped; set OFFICIAL_TEST_JSON to enable it.


[Markdown Cell]
## 13. Preserve the run and interpret results

Save the whole `model_b_run/` output, especially `train_config.json`, `code/`, and
`checkpoints/` including receipts. To resume in a fresh Kaggle session, attach
those checkpoints, keep the same architecture/training settings, and point
`resume_dir` to their directory. Working checkpoints take precedence only when
their committed step is newer. Free disk space must still cover atomic saves.

The default decoder trains on at most 256-token continuations; generation beyond
that range uses less-trained positional embeddings. Inspect held-out token lengths
and increase the target window in a fresh experiment if needed.

The checkpoint budget is 4GiB plus a 1GiB free-space reserve. Two recovery slots
and a full best snapshot are retained; old per-step/per-epoch copies do not accumulate.
Reduce microbatch size in a **new run directory** if GPU memory is insufficient;
automatic OOM retries do not change a running experiment's gradient semantics.
Progress logs and epoch history record NLL, LR, overflows, and processed target tokens.

Conditional validation NLL selects the best checkpoint. The official primary
metric is BERTScore F1; run the supplied evaluator on matching prediction/truth
maps when assessing generation. Document token truncations and architecture
choices in the capstone report. Model A remains on hold; no subsequent model is
created by this notebook.
